In [100]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import os

In [101]:
path = "./raw_data.csv"

In [102]:
df = pd.read_csv(path)

In [103]:
df.columns = (
    df.columns
      .str.strip()
      .str.replace(" ", "_")
)

if "umber_pets" in df.columns:
    df.rename(columns={"umber_pets": "Number_Pets"}, inplace=True)


In [104]:
df.head()

,CustomerID,Region,Gender,Age,Education_Years,Employment_Years,Job_Category,Retired,Marital_Status,Household_Size,...,Coupon_Redemption,Brand_Tenure_Months,Monthly_Spend_ProductA,Cumulative_Spend_ProductA,Monthly_Spend_ProductB,Cumulative_Spend_ProductB,Monthly_Spend_ProductC,Cumulative_Spend_ProductC,Total_Avg_Monthly_Spend,High_Value_Customer
0,0002-GTOKLU-YVY,5,Male,63,16,3,Sales,No,Married,2,...,0,21,$20.60,$84.92,$147.60,$602.92,$176.50,$790.02,$344.70,1
1,0003-RLTRGE-IW2,1,Male,52,11,14,Professional,No,Unmarried,1,...,0,31,$46.20,$304.28,$0.00,$0.00,$107.25,$746.28,$153.45,0
2,0003-UTGKPR-PRU,2,Male,73,14,11,Professional,No,Unmarried,1,...,0,54,$65.20,$703.88,$0.00,$0.00,$0.00,$0.00,$65.20,0
3,0008-ZIQQOT-SGB,3,Female,40,21,3,Sales,No,Married,3,...,0,1,$8.60,$1.72,$171.20,$34.24,$150.25,$36.06,$330.05,1
4,0012-CIVYLF-839,4,Male,40,12,9,Labor,No,Married,2,...,0,48,$46.60,$495.20,$0.00,$0.00,$0.00,$0.00,$46.60,0


In [105]:

text_cols = df.select_dtypes(include="object").columns

for col in text_cols:
    if col in ["CustomerID"]:
        #don't modify the value for CustomerID
        continue
    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
        .str.title()
    )



In [106]:
df.columns

Index(['CustomerID', 'Region', 'Gender', 'Age', 'Education_Years',
       'Employment_Years', 'Job_Category', 'Retired', 'Marital_Status',
       'Household_Size', 'Household_Income', 'Number_Pets', 'Home_Owner',
       'Car_Ownership', 'Car_Brand', 'Car_Value', 'Commute_Distance',
       'Political_Party', 'Votes', 'Credit_Card', 'Credit_Card_Tenure',
       'Active_Lifestyle', 'TV_Watching_Hours', 'Streaming_Svcs',
       'Wireless_Internet', 'Smart_Phone', 'Twitter_Acct', 'LinkedIn_Acct',
       'Facebook_Acct', 'News_Subscriber', 'Coupon_Redemption',
       'Brand_Tenure_Months', 'Monthly_Spend_ProductA',
       'Cumulative_Spend_ProductA', 'Monthly_Spend_ProductB',
       'Cumulative_Spend_ProductB', 'Monthly_Spend_ProductC',
       'Cumulative_Spend_ProductC', 'Total_Avg_Monthly_Spend',
       'High_Value_Customer'],
      dtype='object')

In [107]:
binary_map = {
    "Yes": 1, "No": 0,
    "Y": 1, "N": 0,
    "True": 1, "False": 0,
    "1": 1, "0": 0
}

binary_cols = [
    "Retired",
    "Home_Owner",
    "Active_Lifestyle",
    "Wireless_Internet",
    "Smart_Phone",
    "Twitter_Acct",
    "LinkedIn_Acct",
    "Facebook_Acct",
    "News_Subscriber",
    "Coupon_Redemption",
    "High_Value_Customer",
    "Streaming_Svcs",
    "Votes",
    "Political_Party"
]

for col in binary_cols:
    if col in df.columns:
        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
            .map(binary_map)
        )


In [108]:
currency_cols = [
    "Household_Income",
    "Car_Value",
    "Monthly_Spend_ProductA",
    "Cumulative_Spend_ProductA",
    "Monthly_Spend_ProductB",
    "Cumulative_Spend_ProductB",
    "Monthly_Spend_ProductC",
    "Cumulative_Spend_ProductC",
    "Total_Avg_Monthly_Spend"
]

for col in currency_cols:
    if col in df.columns:
        df[col] = (
            df[col]
            .astype(str)
            .str.replace(r"[$,]", "", regex=True)
        )

numeric_cols = currency_cols + [
    "Age",
    "Education_Years",
    "Employment_Years",
    "Household_Size",
    "Number_Pets"
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

In [109]:
# Validation rules - collect invalid rows into `invalid_df` and keep valid rows in `cleaned_df`
critical_fields = [
    "Age",
    "Household_Income",
    "Total_Avg_Monthly_Spend"
]

In [110]:

# Start with all rows assumed valid
invalid_mask = pd.Series(False, index=df.index)

# 1) Nulls in critical fields -> mark invalid
invalid_mask |= df[critical_fields].isna().any(axis=1)

# 2) Ensure numeric columns are numeric (coercion created NaN for bad values) -> mark those as invalid
if 'numeric_cols' in locals():
    invalid_mask |= df[numeric_cols].isna().any(axis=1)

# 3) Negative values for Education and Employment years are invalid
if 'Education_Years' in df.columns:
    invalid_mask |= df['Education_Years'] < 0
if 'Employment_Years' in df.columns:
    invalid_mask |= df['Employment_Years'] < 0

# 4) Income must be >= 0
if 'Household_Income' in df.columns:
    invalid_mask |= df['Household_Income'] < 0

# 5) Education years must be less than Age
if set(['Education_Years','Age']).issubset(df.columns):
    invalid_mask |= ~(df['Education_Years'] < df['Age'])

# 6) Employment years cannot exceed plausible range: Employment_Years <= Age - 14
if set(['Employment_Years','Age']).issubset(df.columns):
    invalid_mask |= ~(df['Employment_Years'] <= (df['Age'] - 14))

# 7) Car value threshold: if provided and below 1000, mark invalid
if 'Car_Value' in df.columns:
    invalid_mask |= df['Car_Value'].notna() & (df['Car_Value'] < 1000)

# 8) Cumulative spend checks for Products A, B, C: cumulative >= monthly
for p in ['A','B','C']:
    monthly = f'Monthly_Spend_Product{p}'
    cumulative = f'Cumulative_Spend_Product{p}'
    if (monthly in df.columns) and (cumulative in df.columns):
        invalid_mask |= df[cumulative].isna() | df[monthly].isna() | (df[cumulative] < df[monthly])

# 9) Remove duplicate customer records: keep first, mark subsequent duplicates as invalid
if 'CustomerID' in df.columns:
    dup_mask = df.duplicated(subset='CustomerID', keep='first')
    invalid_mask |= dup_mask

# Build invalid and cleaned DataFrames
invalid_df = df[invalid_mask].copy()
cleaned_df = df[~invalid_mask].copy()

# Ensure cleaned_df has no duplicates (keep first) and drop them if present
if 'CustomerID' in cleaned_df.columns:
    cleaned_df = cleaned_df.drop_duplicates(subset='CustomerID', keep='first')


In [111]:
# Create timestamped output directory: outputs/<yy-mm-dd-hh>/
ts = datetime.now().strftime('%y-%m-%d-%H')
outdir = Path('outputs') / ts
outdir.mkdir(parents=True, exist_ok=True)

In [112]:
# Save outputs
cleaned_df.to_csv(outdir / 'customer_data_cleaned.csv', index=False)
invalid_df.to_csv(outdir / 'invalid.csv', index=False)

print('Saved', cleaned_df.shape[0], 'clean rows and', invalid_df.shape[0], 'invalid rows to', outdir)

Saved 4137 clean rows and 863 invalid rows to outputs\26-04-13-22
